# Open-book test
A small open model answers questions about a made-up rulebook it has never seen: with no rulebook, with the whole rulebook, with 2 pages found by matching words, and with 2 pages found by meaning.

Runs on a free Colab GPU (Runtime → Change runtime type → T4 GPU), or slowly on a CPU. Answers can differ a little from run to run on different hardware.

In [ ]:
!pip -q install transformers accelerate

In [ ]:
import re, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED = "sentence-transformers/all-MiniLM-L6-v2"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if dev == "cuda" else torch.float32).to(dev).eval()
etok, emodel = AutoTokenizer.from_pretrained(EMBED), AutoModel.from_pretrained(EMBED).eval()

In [ ]:
RULEBOOK = """
# Student Halls — House Rules

(A made-up university hall of residence. No real university or company.)

## 1. The hall office
The hall office is in room G04 on the ground floor. It is open on Tuesdays and Thursdays from 4 pm to 7 pm. Outside those hours, write to the office by email and expect a reply within two working days.

## 2. Paying rent
Rent is due on the 3rd day of each month and is paid by bank transfer. If the money has not arrived by the 10th, a late fee of 35 euros is added to the next month's rent.

## 3. Your deposit
Your deposit is returned within 21 days after you hand back all your keys. The office checks the room first, and the cost of any cleaning or damage beyond normal wear is taken from the deposit.

## 4. Moving out
To end your tenancy you must give nine weeks' written notice. Send the notice by email to the hall office; a phone call does not count as notice.

## 5. Quiet hours
Quiet hours are from 10:30 pm to 7 am on weeknights, and from 11:30 pm to 9 am on Friday and Saturday nights. During quiet hours, keep music, televisions and voices at a level that cannot be heard in the next room.

## 6. Animals
Each room may keep one cat, or one dog that weighs less than 8 kg when fully grown. You need written permission from the hall office before the animal moves in. Birds and other animals are not allowed.

## 7. Bicycles
Bicycles are kept in basement room B2. The door code is given out by the hall office. Bicycles may not be left in hallways, on stairs or in bedrooms.

## 8. Rubbish and recycling
The bins are behind door C in the courtyard. Glass is collected on Tuesdays and paper on Fridays. General rubbish is collected every Monday and Thursday morning.

## 9. Laundry room
The shared washing machines and dryers are on the fourth floor and can be used from 7 am to 9 pm. Machines run on tokens from the hall office; one wash costs two tokens and one dry costs one token.

## 10. Overnight guests
Guests may stay overnight for up to 14 nights in a row. A guest who stays longer must be registered with the hall office.

## 11. Parking
The building has 12 parking spaces, given out from a waiting list kept by the hall office. The two visitor spaces by the gate may be used for up to 3 hours at a time.

## 12. Heating
The central heating is switched on from 15 October to 15 April. Each radiator has its own dial; please do not cover radiators with furniture or curtains.

## 13. Repairs
Report repairs with the form on the office door or by email. The office replies within three working days. For an emergency such as a water leak, a burst pipe or no power, call the 24-hour repair line printed on the notice board in the entrance hall.

## 14. Smoking
Smoking is not allowed anywhere inside the building, including bedrooms, stairwells and the lift. Residents may smoke in the corner of the courtyard by the oak tree.

## 15. Barbecues
Charcoal and gas barbecues are not allowed anywhere in the hall or its garden. Small electric grills are allowed in the shared kitchens.

## 16. Decorating
You may put up posters with removable putty only. Nails, screws and tape on the walls are not allowed.

## 17. Subletting
You may not sublet your room without written permission from the hall office. Short holiday rentals to tourists are not allowed at all.

## 18. Keys
A lost key costs 60 euros to replace. If you are locked out outside office hours, you must call a locksmith yourself and pay the call-out fee.

## 19. Parcels
Deliveries that do not fit in your letterbox are kept in the parcel room, G02, for 10 days. After 10 days they are returned to the sender.

## 20. Moving in and moving day
Book the lift for moving through the hall office. A booking lasts up to 4 hours and is only possible on Saturdays between 9 am and 5 pm.

## 21. Common room
The common room on the ground floor can be booked up to two weeks ahead and used until 10 pm. A cleaning deposit of 50 euros is paid when you book and returned if the room is left clean.

## 22. Fire alarm
The fire alarm is tested at 10 am on the first Monday of every month. If it rings at any other time, leave the building by the stairs, not the lift.

## 23. Windows
The outsides of all windows are cleaned twice a year, in April and in October. You will get a note a week before.

## 24. Insurance
Every resident must have contents insurance for their own belongings. The building's insurance covers the building only, not your things.

## 25. Roof terrace
The roof terrace is open from May to September, between 8 am and 10 pm. Glass bottles are not allowed on the terrace.

## 26. Snow and ice
The building clears snow from the paths and the courtyard. Residents clear snow from their own window ledges.

## 27. Noise complaints
If a neighbour is too loud, first speak to them politely. If that does not help, send a written complaint to the hall office with the dates and times.

## 28. Internet in the common room
There is free Wi-Fi in the common room. The password is changed on the first day of each month and posted on the common room door.
"""

In [ ]:
STOP = set("a an the of to in on for and or is are be do does i my me can what when where which how much many with at by it its this that you your will from up out after before there any all not as if".split())
words = lambda s: {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP}

def split_sections(text):
    return {i + 1: b.strip() for i, b in enumerate(re.split(r"\n(?=## )", text.strip())) if b.strip().startswith("## ")}

def embed(texts):
    enc = etok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = emodel(**enc).last_hidden_state
    m = enc.attention_mask.unsqueeze(-1).float()
    return torch.nn.functional.normalize((out * m).sum(1) / m.sum(1), dim=1).numpy()

def load(text):
    global SECTIONS, WHOLE, IDS, VEC
    SECTIONS = split_sections(text)
    WHOLE = "\n\n".join(SECTIONS.values())
    IDS = list(SECTIONS)
    VEC = embed([SECTIONS[n] for n in IDS])

def keyword_top(q, k=2):
    score = {n: len(words(q) & words(s)) for n, s in SECTIONS.items()}
    return sorted(score, key=lambda n: (-score[n], n))[:k]

def meaning_top(q, k=2):
    sims = VEC @ embed([q])[0]
    return [IDS[i] for i in np.argsort(-sims)[:k]]

def ask(system, q):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": q}]
    ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").input_ids.to(dev)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=90, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

pages = lambda ns: "\n\n".join(SECTIONS[n] for n in ns)
CLOSED = "You help students living in a university hall of residence. Answer in one or two short sentences."
BOOK = "You help students living in a university hall of residence. Answer in one or two short sentences, using the handbook pages below.\n\n{pages}"
load(RULEBOOK)

In [ ]:
QUESTIONS = [
    ("Can my puppy live with me in my room?", 6, r"\b8\b|eight|permission"),
    ("How much warning do I have to give before I move out?", 4, r"nine|\b9\b"),
    ("When will I get my deposit back after I leave?", 3, r"\b21\b|twenty-one"),
    ("Until what time can I play music on a Saturday night?", 5, r"11:30|11\.30|23:30"),
    ("Where do I keep my bike?", 7, r"B2"),
    ("Which day do I put out my empty jam jars?", 8, r"Tuesday"),
    ("How many tokens does one wash cost?", 9, r"\btwo\b|\b2\b"),
    ("My friend wants to stay with me for three weeks. Is that OK?", 10, r"\b14\b|fourteen|regist"),
    ("When does the heating come on?", 12, r"15 October|October 15"),
    ("Can I smoke in my bedroom?", 14, r"\bnot\b|\bno\b|n't"),
    ("Can I use a charcoal barbecue in the garden?", 15, r"\bnot\b|\bno\b|n't"),
    ("I lost my key. How much will a new one cost?", 18, r"\b60\b|sixty"),
    ("How long will the building hold a delivery for me?", 19, r"\b10\b|ten"),
    ("When is the fire alarm tested?", 22, r"first Monday"),
    ("Can I rent my room out to tourists for a weekend?", 17, r"\bnot\b|\bno\b|n't"),
    ("Do I need insurance for my own belongings?", 24, r"\byes\b|must|required|need"),
    ("A pipe burst in my kitchen at midnight. What do I do?", 13, r"24-hour|24 hour|repair line"),
    ("When is the hall office open?", 1, r"Tuesday"),
    ("How long can a visitor park outside?", 11, r"\b3\b|three"),
    ("When can I use the roof terrace?", 25, r"May"),
]

NOT_IN_BOOK = [
    "Is there a gym in the building?",
    "How much is the rent for a double room?",
    "Can I install a satellite dish?",
    "Are electric scooters allowed in the lift?",
    "What is the phone number of the 24-hour repair line?",
]

In [ ]:
for q, right_page, _ in QUESTIONS:
    kw, mn = keyword_top(q), meaning_top(q)
    print("Q:", q, "  | the right page:", right_page)
    print("  no rulebook   :", ask(CLOSED, q))
    print("  whole rulebook:", ask(BOOK.format(pages=WHOLE), q))
    print("  word search   ", kw, ":", ask(BOOK.format(pages=pages(kw)), q))
    print("  meaning search", mn, ":", ask(BOOK.format(pages=pages(mn)), q))
    print()

In [ ]:
for q in NOT_IN_BOOK:
    mn = meaning_top(q)
    print("Q:", q, mn)
    print("  ", ask(BOOK.format(pages=pages(mn)), q), "\n")

## Try your own document

In [ ]:
MY_TEXT = """## 1. First section
Paste your own text here.

## 2. Second section
Start each section with a line beginning "## ".
"""
load(MY_TEXT)
q = "Your question here?"
print("pages found by meaning:", meaning_top(q))
print(ask(BOOK.format(pages=pages(meaning_top(q))), q))